In [ ]:
import torch 
import numpy
import matplotlib.pyplot as plt
import time
from os import listdir, path
from tqdm import tqdm
from torch.utils.data import DataLoader

import datetime
import sys 
from torch import nn
sys.path.append('../../datasets/')
from sequences import SequencesDataset
# from datasets.sequences import  SequencesDataset
import glob
import json
import math
import numpy as np
import torch
from torch.utils.data import Dataset
from os import listdir, path
import sys

import argparse
import datetime
import os
import time
from os import path

import numpy as np
import torch
# from datasets.sequences import SequencesDataset
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from models.simple_CNN import CNN




In [ ]:
order = 2
exposure = 20
seq_length = 1
train_dataset = SequencesDataset(exposure = exposure, polynomial_order = order,split = 'train', sequence_length = seq_length)
test_dataset = SequencesDataset(exposure = exposure, polynomial_order = order, split= 'test', sequence_length = seq_length)
dev_dataset = SequencesDataset(exposure = exposure, polynomial_order = order, split = 'validation', sequence_length = seq_length)


In [ ]:
train_dataloader =  DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader =  DataLoader(test_dataset, batch_size=32, shuffle=True)
dev_dataloader =  DataLoader(dev_dataset, batch_size=32, shuffle=True)

In [ ]:
os.path.join(os.getcwd(), '..')

In [ ]:
import datetime
import os
now = datetime.datetime.now()
datetime_string = now.strftime("%Y_%m_%d_%H_%M_%S")
if os.getcwd().endswith('notebooks'):
    results_folder = path.join(os.getcwd(), '../../', f'runs/simple_CNN/exposure_{exposure}/order_{order}_frames_{seq_length}/{datetime_string}')
else:
    results_folder = 'runs/simple_CNN/exposure_{exposure}/order_{order}_frames_{seq_length}/{datetime_string}'
os.makedirs(results_folder, exist_ok=True)

In [ ]:
results_folder = '/media/cmu/DATA1/francis/Melt_Pool_Data/ExperimentalDepthPrediction/runs/simple_CNN/exposure_20/order_2_frames_1/2023_04_29_17_02_54'
exposure = results_folder.split('exposure_')[-1].split(os.sep)[0]
order = results_folder.split('order_')[-1].split('frames_')[-1].split(os.sep)[0]
seq_length = results_folder.split('frames_')[-1].split(os.sep)[0]
print(exposure, order, seq_length)

results_folder[0]
model = CNN(seq_length, 2*(order + 1))

print(fr"Now training: Order: {order}, Exposure: {exposure}µs, Image Type: {image_type}, Frames: {seq_length}")
train_dataset = SequencesDataset(exposure = exposure, polynomial_order = order,image_type = image_type, split = 'train', sequence_length = seq_length)
test_dataset = SequencesDataset(exposure = exposure, polynomial_order = order,image_type = image_type,  split= 'test', sequence_length = seq_length)
dev_dataset = SequencesDataset(exposure = exposure, polynomial_order = order, image_type = image_type, split = 'validation', sequence_length = seq_length)


train_dataloader =  DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader =  DataLoader(test_dataset, batch_size=32, shuffle=True)
dev_dataloader =  DataLoader(dev_dataset, batch_size=32, shuffle=True)

#     optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
#     from torch.optim.lr_scheduler import StepLR
#     scheduler= StepLR(optimizer, step_size = 10, gamma = 0.5)
#     model_fname = os.path.join(diffusion_results_dir, 'ckpt.pth')
#     model_checkpoint = torch.load(model_fname)
#     model.load_state_dict(model_checkpoint[0])
#     optimizer.load_state_dict(model_checkpoint[1])

#     model.eval()

#     for param in model.parameters():
#         param.requires_grad = False
#     return model

In [ ]:
device = 'cuda'
criterion  = nn.MSELoss()
learning_rate = 1e-4
losses =[]
test_losses = []
model = CNN(seq_length, 2*(order + 1))
model.to(device)
plotting_interval = 5
epochs = 1000
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate,weight_decay = 0)
for epoch in tqdm(range(epochs)):
    model.train()

    running_loss = 0
    test_running_loss = 0
    running_scaled_loss = 0
    start_time = time.time()
    model.train()
    for batch_num, (image, target) in enumerate(train_dataloader):
        image = image.to(device)
        target = target.to(device)
        output = model((image.float()))
        loss = criterion(target.float(), (output.float()))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()/len(train_dataloader)
    
    losses.append(running_loss)
    model.eval()
    for batch_num, (image, target) in enumerate(dev_dataloader):
        image = image.to(device)
        target = target.to(device)
        output = model((image.float()))
        test_loss = criterion(target.float(), (output.float()))
        test_running_loss += test_loss.item()/len(test_dataloader)

    test_losses.append(test_running_loss)

    if epoch % 20 == 0:
        print(f'Epoch [{epoch}/{epochs}]Train Loss: {losses[-1]}, Validation Loss: {test_losses[-1]}')
        np.savetxt(os.path.join(results_folder, 'test_loss.txt'), test_losses)
        np.savetxt(os.path.join(results_folder, 'train_loss.txt'), losses)
        states = [
            model.state_dict(),
            optimizer.state_dict(),
            epoch,
            batch_num,
        ]
        torch.save(states, os.path.join(results_folder, "ckpt.pth"))


In [ ]:
def decode_coeffs(coefficients, resolution = 100):
    t_new = np.linspace(0, 1, resolution)
    x_new = np.poly1d(coefficients[:len(coefficients)//2])
    y_new = np.poly1d(coefficients[len(coefficients)//2:])
    return x_new(t_new), y_new(t_new)


In [ ]:
for batch_num, (image, target) in enumerate(train_dataloader):
    # breakpoint()
    # print('hr', hr.shape)

    image = image.to(device)
    target = target.to(device)
    output = model((image.float()))
    print(nn.L1Loss()(output, target))
    for i in range(image.shape[0]):
      pred = model(image.to(device).float())[i].cpu().detach().numpy()
      gt = target[i].cpu().detach().numpy()
      plt.plot(pred)
      plt.plot(gt)
      plt.show()
      pred_curve_x, pred_curve_y =decode_coeffs(pred)
      gt_curve_x, gt_curve_y = decode_coeffs(gt)
      plt.plot(pred_curve_x, -pred_curve_y)
      plt.plot(gt_curve_x, -gt_curve_y)
      print(np.sum(np.abs(pred-gt)))
      plt.show()
    break

In [ ]:
for batch_num, (image, target) in enumerate(train_dataloader):
    # breakpoint()
    # print('hr', hr.shape)

    image = image.to(device)
    target = target.to(device)
    output = model((image.float()))
    print(nn.L1Loss()(output, target))
    for i in range(image.shape[0]):
      pred = model(image.to(device).float())[i].cpu().detach().numpy()
      gt = target[i].cpu().detach().numpy()
      plt.plot(pred)
      plt.plot(gt)
      plt.show()
      pred_curve_x, pred_curve_y =decode_coeffs(pred)
      gt_curve_x, gt_curve_y = decode_coeffs(gt)
      plt.plot(pred_curve_x, -pred_curve_y)
      plt.plot(gt_curve_x, -gt_curve_y)
      print(np.sum(np.abs(pred-gt)))
      plt.show()
    break

In [ ]:
t_new = np.linspace(0, 1, 100)
x_new = np.poly1d(model(input.to(device).float())[5].cpu().detach().numpy()[:5])
y_new = np.poly1d(model(input.to(device).float())[5].cpu().detach().numpy()[5:])

x_new = np.poly1d(target[i].cpu().detach().numpy()[:5])
y_new = np.poly1d(target[i].cpu().detach().numpy()[5:])
# plt.plot(target[i].cpu().detach().numpy())
x_new_pred = np.poly1d(model(input.to(device).float())[5].cpu().detach().numpy()[:5])
y_new_pred = np.poly1d(model(input.to(device).float())[5].cpu().detach().numpy()[5:])
plt.plot(x_new_pred(t_new), y_new_pred(t_new), label="Polynomial Fit (Order = 7)")

plt.title("Polynomial Fit (Order = 7) to Measured Melt Pool Contour")
plt.plot(x_new(t_new), y_new(t_new), label="Polynomial Fit (Order = 7)")
# plt.plot(set_5_1_bead_1_df.X, -set_5_1_bead_1_df.Y, label="Measured Contour")
plt.axis("off")
plt.legend()

In [ ]:
torch.sum((x_new_pred(t_new) - x_new(t_new))**2 + (y_new_pred(t_new) - y_new(t_new))**2)

In [ ]:
plt.plot(x_new_pred(t_new), label="Polynomial Fit (Order = 7)")
plt.plot(x_new(t_new), label="Polynomial Fit (Order = 7)")



In [ ]:
plt.plot(y_new_pred(t_new), label="Polynomial Fit (Order = 7)")
plt.plot(y_new(t_new), label="Polynomial Fit (Order = 7)")


In [ ]:
def reconstruct(t,coeffs):
    print(t.shape)
    order = (coeffs.shape[1])//2
    x_coeffs = coeffs[:,:(coeffs.shape[1])//2]
    y_coeffs = coeffs[:,(coeffs.shape[1])//2:]
    result_x = torch.zeros(t.shape).float().to(device)
    result_y = torch.zeros(t.shape).float().to(device)

    for idx,i in enumerate(range(order)):
        order = len(coeffs)//2 -1 -idx
        # print(result_x)
        print(y_coeffs.shape)
        result_x += x_coeffs[:,idx:idx+1]*t**order
        print(y_coeffs[:,idx:idx+1].shape, t.shape)
        result_y += y_coeffs[:,idx:idx+1]*t**order
        
    return result_x, result_y
